# 02. Validación de la corrida

Última tarea del job. Hace tres cosas: muestra los conteos de cada tabla de Bronze, Silver y Gold, muestra los registros rechazados por fuente (desde `silver.cuarentena`) y **falla si alguna tabla de Gold queda vacía**. Además guarda los conteos en `default.conteos_por_corrida`, que es la tabla de donde salen los conteos por lote del README.

In [ ]:
dbutils.widgets.text("catalogo", "electrocasa_dev")
CATALOGO = dbutils.widgets.get("catalogo").strip()
if not CATALOGO or not all(c.isalnum() or c == "_" for c in CATALOGO):
    raise ValueError("El catálogo solo admite letras, números y guion bajo")

In [ ]:
dbutils.widgets.text("job_run_id", "manual")
JOB_RUN_ID = dbutils.widgets.get("job_run_id").strip()
if not JOB_RUN_ID or not all(c.isalnum() or c in "_-" for c in JOB_RUN_ID):
    raise ValueError("job_run_id solo admite letras, números, guion y guion bajo")

In [ ]:
TABLAS = {
    "bronze": ["ventas", "catalogo", "empleados", "resenas", "devoluciones", "sucursales", "tracking_envios"],
    "silver": ["ventas", "catalogo_snapshots", "catalogo", "empleados_eventos", "empleados_historia", "resenas",
               "devoluciones", "tracking_envios", "sucursales", "cuarentena", "dni_duplicados"],
    "gold": ["ventas_netas_sucursal_mes", "top_productos_unidades", "top_productos_devolucion", "dotacion_sucursal_mes",
             "ventas_vendedor_mes", "resenas_negativas_categoria_mes", "resenas_negativas_origen_mes",
             "despachos_courier", "empleados_vigentes"],
}

In [ ]:
from pyspark.sql import functions as F

conteos = []
for capa, tablas in TABLAS.items():
    for tabla in tablas:
        conteos.append((capa, tabla, spark.table(f"{CATALOGO}.{capa}.{tabla}").count()))
df_conteos = spark.createDataFrame(conteos, "capa STRING, tabla STRING, filas BIGINT")
display(df_conteos)

In [ ]:
rechazos = spark.sql(f"""
    SELECT fuente, count(*) AS rechazados
    FROM {CATALOGO}.silver.cuarentena
    GROUP BY fuente
    ORDER BY fuente
""")
display(rechazos)
display(spark.sql(f"""
    SELECT fuente, regla, count(*) AS registros
    FROM {CATALOGO}.silver.cuarentena
    LATERAL VIEW explode(reglas_fallidas) AS regla
    GROUP BY fuente, regla
    ORDER BY fuente, regla
"""))

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOGO}.default.conteos_por_corrida (
      job_run_id STRING, fecha_validacion TIMESTAMP, capa STRING, tabla STRING, filas BIGINT
    ) COMMENT 'Conteos que deja cada corrida del job, para comparar lote por lote'
""")
registro = (df_conteos
            .unionByName(rechazos.select(F.lit("cuarentena").alias("capa"), F.col("fuente").alias("tabla"),
                                         F.col("rechazados").alias("filas")))
            .withColumn("job_run_id", F.lit(JOB_RUN_ID))
            .withColumn("fecha_validacion", F.current_timestamp())
            .select("job_run_id", "fecha_validacion", "capa", "tabla", "filas"))
registro.write.mode("append").saveAsTable(f"{CATALOGO}.default.conteos_por_corrida")

In [ ]:
vacias = [tabla for capa, tabla, filas in conteos if capa == "gold" and filas == 0]
if vacias:
    raise Exception(f"Validación fallida: tablas de Gold vacías: {', '.join(vacias)}")
print("Validación correcta: todas las tablas de Gold tienen filas")

## Conteos por lote para el README

Después de procesar los tres lotes, esta consulta muestra lado a lado los conteos de cada corrida.

In [ ]:
display(spark.sql(f"""
    SELECT capa, tabla, job_run_id, filas
    FROM {CATALOGO}.default.conteos_por_corrida
    ORDER BY capa, tabla, fecha_validacion
"""))